In [1]:
import ot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import math
import scipy
from scipy.spatial import cKDTree
from scipy import integrate
from scipy import interpolate
from scipy.spatial import Delaunay
from scipy.spatial import Voronoi
from itertools import product
import torch
torch.manual_seed(100000)
import torch.nn as nn
import torch.optim as optim


In [2]:
#settings
N = 5  # Number of variables
F = 8  # Forcing
start = F * np.ones(N)  # Initial state (equilibrium)
start[0] += 0.01  # Add small perturbation to the first variable
dt = 0.01
subset_size = 200#cell number
traj_length = int(1e6)#length of trajectory
noise_level = 0.2#.3
sample_size = int(5000)
slope = 0.05
simulation_t = int(5e3)
epsilon = 1e-4
steps = 5
N_iters = 20000
costs1W, costs2W, costs1L, costs2L = [],[],[],[]
def L96(x, t):
    """Lorenz 96 model with constant forcing"""
    return (np.roll(x, -1) - np.roll(x, 2)) * np.roll(x, 1) - x + F 
def L96_vec_unforced(X):

    return (torch.roll(X, shifts=-1, dims=1) - torch.roll(X, shifts=2, dims=1)) * torch.roll(X, shifts=1, dims=1) - X 
t = np.arange(0.0, traj_length*dt, dt)

def invariant_measure(matrix):
    N = len(matrix)
    rhs = (-1) * (epsilon/N) * torch.ones(N)
    rho = torch.linalg.solve(((1 - epsilon) * matrix - torch.eye(N)),rhs)
    return rho

relu = nn.ReLU()
def decay(x):
    return relu(1-slope*x)
def w(xs):
    dists =  torch.cdist(xs, Voronoi_centers, p =2)
    pre_w = decay(dists)
    return  pre_w / torch.sum(pre_w, dim=1,keepdim = True)


def Ulam(points,Tpoints):
    mat = torch.zeros((subset_size, subset_size))
    #before normalization
    with torch.no_grad():
          randpts_idxs = torch.tensor(tree.query(points.detach().numpy())[1], dtype=torch.int)
    weights = w(Tpoints)
    mat.index_add_(0, randpts_idxs, weights)
    mat = (mat.T)/mat.sum(dim = 1)
    return mat
class W2Loss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, im_net):
        im_net_np = im_net.detach().numpy()
        im_net_np /= np.sum(im_net_np)
        im_true_np_norm = im_true_np / np.sum(im_true_np)  
        costM = ot.dist(np.arange(subset_size).reshape(-1, 1), 
                        np.arange(subset_size).reshape(-1, 1))
        _, log = ot.emd(im_true_np_norm, im_net_np, costM, log=True)
        cost,grad = log['cost'],log["v"]
        loss = np.sum(cost)
        grad_tensor = torch.tensor(grad, dtype=im_net.dtype)
        ctx.save_for_backward(grad_tensor)
        return torch.tensor(loss, dtype=U_net.dtype)
    @staticmethod
    def backward(ctx, grad_output):
        grad_tensor, = ctx.saved_tensors  # Unpack gradient
        return grad_tensor.reshape(subset_size,)

In [3]:
trajectory_clean = scipy.integrate.odeint(L96, start, t)
trajectory = trajectory_clean + np.random.normal(0,noise_level,((traj_length,5)))#loooooooong trajectory
trajectory = trajectory[int(1e3):]
rand_idxs = np.random.choice(len(trajectory), size=sample_size, replace=False)
observed = trajectory[rand_idxs]
randpts = torch.tensor(trajectory[rand_idxs],dtype = torch.float)
Trandpts = torch.tensor(trajectory[rand_idxs+steps],dtype = torch.float)
#normalization
M_scale = torch.max(torch.abs(randpts))
randpts /= M_scale
Trandpts /= M_scale

#Voronoi cell center
Voronoi_centers = MiniBatchKMeans(n_clusters=subset_size).fit(randpts).cluster_centers_

tree = cKDTree(Voronoi_centers)
Voronoi_centers = torch.tensor(Voronoi_centers,dtype = torch.float)
U_true = Ulam(randpts,Trandpts)
im_true = invariant_measure(U_true)
im_true_np = im_true.detach().cpu().numpy()

/homes/yinonghyn/.local/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)


In [ ]:
for iteration in range(10):
    net1 = nn.Sequential(
            nn.Linear(5, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 1))

    optimizer1 = optim.Adam(net1.parameters(),lr = 1e-3)

    net1.train()
    loss1 = []
    net1_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field = L96_vec_unforced(net1_randpts*M_scale)/M_scale
        V_field += net1(net1_randpts)
        net1_randpts = net1_randpts+dt* V_field
    U_net = Ulam(randpts,net1_randpts)
    im_net = invariant_measure(U_net)
    initial_L1 =  torch.linalg.norm(im_net - im_true, dtype=torch.float32)
    for i in range(N_iters):
        optimizer1.zero_grad()
        net1_randpts = randpts
        for _ in range(steps):
            V_field = L96_vec_unforced(net1_randpts*M_scale)/M_scale
            V_field += net1(net1_randpts)
            net1_randpts = net1_randpts+dt* V_field
        U_net = Ulam(randpts,net1_randpts)
        im_net = invariant_measure(U_net)
        L1 = torch.linalg.norm(im_net - im_true, dtype=torch.float32)
        L1.backward()
        optimizer1.step()
        loss1.append(L1.item())
        with torch.no_grad():
            if i % 500 == 0:
                if L1.item() < 0.25 * initial_L1:
                    break
    x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, N)
    vals1 = [x1.detach().numpy().flatten()]
    for _ in range(simulation_t-1):
        V_field = L96_vec_unforced(x1*M_scale)/M_scale
        V_field += net1(x1)
        x1 = x1+dt* V_field
        vals1.append(x1.detach().numpy().flatten())
    vals1 = np.array(vals1)
    net2 = nn.Sequential(
            nn.Linear(5, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 1))


    optimizer2 = optim.Adam(net2.parameters(),lr = 1e-3)
    net2.train()
    loss2 = []
    net2_randpts = randpts.clone()  # safe copy, avoid modifying original tensor
    for _ in range(steps):
        V_field2 = L96_vec_unforced(net2_randpts*M_scale)/M_scale
        V_field2 += net2(net2_randpts)
        net2_randpts = net2_randpts+dt* V_field2
    initial_L2 = torch.mean((net2_randpts - Trandpts) ** 2)
    for i in range(N_iters):
    # Zero gradients for each optimizer
        optimizer2.zero_grad()
        net2_randpts = randpts
        for _ in range(steps):
            V_field2 = L96_vec_unforced(net2_randpts*M_scale)/M_scale
            V_field2 += net2(net2_randpts)
            net2_randpts = net2_randpts+dt* V_field2

        L2 = torch.mean((net2_randpts - Trandpts) ** 2)
        L2.backward()
        optimizer2.step()
        loss2.append(L2.item())
        with torch.no_grad():
            if i % 500 == 0:
                if L2.item() < 0.25 * initial_L2:
                    break
    x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 5)
    vals2 = [x2.detach().numpy().flatten()]
    for _ in range(simulation_t-1):
        V_field2 = L96_vec_unforced(x2*M_scale)/M_scale
        V_field2 += net2(x2)
        x2 = x2+dt* V_field2

        vals2.append(x2.detach().numpy().flatten())
    vals2 = np.array(vals2)
    GT = scipy.integrate.odeint(L96, M_scale*randpts[0], np.arange(0.0, simulation_t*dt, dt))#[::steps]
    
    
    subGT, subvals1, subvals2 = GT, np.array(M_scale * vals1), np.array(M_scale * vals2)
    a, b = np.ones((simulation_t,)) / simulation_t, np.ones((simulation_t,)) / simulation_t
    M1, M2 = ot.dist(subGT, subvals1),ot.dist(subGT, subvals2)
    _,G01 = ot.emd(a, b, M1,log = 'true')
    _,G02 = ot.emd(a, b, M2,log = 'true')
    cost1W,cost2W = (G01['cost'])**(1/2),(G02['cost'])**(1/2)
    GTpts = torch.tensor((GT/M_scale)[:-steps],dtype = torch.float)
    GTTpts = GT[steps:]
    net1pts = GTpts.clone()
    net2pts = GTpts.clone()

    for i in range(steps):
        V_unforced1 = L96_vec_unforced(net1pts * M_scale) / M_scale
        V_unforced2 = L96_vec_unforced(net2pts * M_scale) / M_scale

        Vf1 = V_unforced1 + net1(net1pts)
        Vf2 = V_unforced2 + net2(net2pts)

        net1pts = net1pts + dt * Vf1
        net2pts = net2pts + dt * Vf2
    cost1L = (np.mean((np.array(M_scale)*net1pts.detach().numpy()-GTTpts)**2))**(1/2)
    cost2L = (np.mean((np.array(M_scale)*net2pts.detach().numpy()-GTTpts)**2))**(1/2)
    print("Experiment ", iteration,": W2(ours) ",cost1W, ", W2(pointwise) ",cost2W)
    print("Experiment ", iteration,": L2(ours) ",cost1L, ", L2(pointwise) ",cost2L)
    costs1W.append(cost1W)
    costs2W.append(cost2W)
    costs1L.append(cost1L)
    costs2L.append(cost2L)

/tmp/ipykernel_796136/2001721527.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, N)
/tmp/ipykernel_796136/2001721527.py:84: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x2 = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 5)
/homes/yinonghyn/.local/lib/python3.10/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)
/tmp/ipykernel_796136/2001721527.py:102: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True

Experiment  0 : W2(ours)  2.913090548544077 , W2(pointwise)  2.919623143093976
Experiment  0 : L2(ours)  0.04661373075314656 , L2(pointwise)  0.10905523904319318
Experiment  1 : W2(ours)  2.573594731623855 , W2(pointwise)  2.4841564511232326
Experiment  1 : L2(ours)  0.044158613682052456 , L2(pointwise)  0.11991160723431928


In [ ]:
costs1W,costs2W,costs1L,costs2L

In [ ]:
np.mean(costs1W),np.std(costs1W)

In [ ]:
np.mean(costs2W),np.std(costs2W)

In [ ]:
np.mean(costs1L),np.std(costs1L)

In [ ]:
np.mean(costs2L),np.std(costs2L)